# NLP Masterclass 04: Sequence-to-Sequence & the Attention Mechanism

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 03 (RNNs & LSTMs)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"The Bahdanau Attention mechanism (2014) allowed the decoder to 'look back' at the encoder. The Transformer (2017) removed the encoder's recurrence entirely, using ONLY attention. What made the authors believe that attention alone was sufficient — what proof did they have?"*

---

## The Evolutionary Bridge: RNN → Attention → Transformer

This notebook is the **missing link** between RNNs (NLP 03) and Transformers (DL 07).

| Year | Architecture | Limitation Solved |
|------|-------------|------------------|
| 2014 | **Seq2Seq** (Sutskever) | Variable-length input → variable-length output |
| 2015 | **+ Attention** (Bahdanau) | Bottleneck of fixed-size context vector |
| 2017 | **Transformer** (Vaswani) | Sequential processing bottleneck |

## Table of Contents
1. The Encoder-Decoder Framework
2. The Information Bottleneck Problem
3. Bahdanau Attention from Scratch
4. From RNN-Attention to Self-Attention

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors see attention as "just another neural network component." Seniors understand that attention is a **differentiable database query** — Query finds relevant Keys, retrieves corresponding Values. This same abstraction appears in RAG (NB28: query=user question, keys/values=document chunks) and Agentic tool selection (NB30: query=task, keys=tool descriptions).

**Mental Model:** Imagine translating a 100-word English sentence to French. Without attention, the encoder must compress the entire sentence into ONE vector (the context). That's like reading a book, then trying to recite it from a single sticky note. With attention, the translator can look back at any word at any time — like keeping the original book open while translating.

**Common Junior Pitfall:** Thinking attention is only for NLP. Attention is used in computer vision (ViT), protein folding (AlphaFold), music generation, and graph neural networks.

---

In [ ]:
!pip install -q torch numpy matplotlib

## 1 · The Encoder-Decoder Framework

**Problem:** Map sequences of different lengths.
- Translation: "Hello world" (2 words) → "Bonjour le monde" (3 words)
- Summarization: 500 words → 50 words
- Chatbots: Question → Answer

```
ENCODER                              DECODER
[Hello] → [RNN] → h₁ ─┐             ┌─ h₁' → [RNN] → "Bonjour"
[world] → [RNN] → h₂ ─┤  context    ├─ h₂' → [RNN] → "le"
                       └──→ c ───→──┘  h₃' → [RNN] → "monde"
                      (bottleneck!)     h₄' → [RNN] → <EOS>
```

### The Bottleneck Problem
ALL information from the encoder must pass through ONE vector (`c`). For long sequences, this vector can't hold enough information.

In [ ]:
import torch
import torch.nn as nn

class Seq2SeqEncoder(nn.Module):
    """Encoder: reads input sequence, produces hidden states."""
    
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
    
    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        # outputs: ALL hidden states (seq_len, hidden_dim)
        # hidden: LAST hidden state (the "context" vector)
        return outputs, hidden, cell

# Demo
encoder = Seq2SeqEncoder(vocab_size=5000, embed_dim=64, hidden_dim=128)
src = torch.randint(0, 5000, (2, 8))  # Batch=2, source seq_len=8
enc_outputs, enc_hidden, enc_cell = encoder(src)

print(f'Source sequence:  {src.shape}')
print(f'Encoder outputs:  {enc_outputs.shape}  (ALL 8 hidden states)')
print(f'Encoder hidden:   {enc_hidden.shape}  (LAST hidden state = context)')
print(f'\nWithout attention: decoder only gets enc_hidden (128 numbers for 8 words!)')
print(f'With attention: decoder can access ALL enc_outputs at every step')

---
## 2 · Bahdanau Attention from Scratch

### The Key Insight (Bahdanau et al., 2015)

Instead of compressing the entire input into one vector, let the decoder **dynamically decide which encoder states to focus on** at each decoding step.

$$\text{score}(h_t^{dec}, h_j^{enc}) = v^T \tanh(W_1 h_t^{dec} + W_2 h_j^{enc})$$
$$\alpha_{t,j} = \text{softmax}(\text{score}_{t}) \quad \text{(attention weights)}$$
$$c_t = \sum_j \alpha_{t,j} h_j^{enc} \quad \text{(context vector — different for each decoder step!)}$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BahdanauAttention(nn.Module):
    """Additive (Bahdanau) attention mechanism.
    
    This is the original attention that inspired the Transformer.
    The self-attention in DL 07 is the Scaled Dot-Product variant.
    """
    
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)  # For decoder state
        self.W2 = nn.Linear(hidden_dim, hidden_dim)  # For encoder outputs
        self.v = nn.Linear(hidden_dim, 1)             # Score vector
    
    def forward(self, decoder_hidden, encoder_outputs):
        """
        Args:
            decoder_hidden: (batch, hidden_dim) — current decoder state
            encoder_outputs: (batch, src_len, hidden_dim) — all encoder states
        Returns:
            context: (batch, hidden_dim) — weighted sum of encoder states
            weights: (batch, src_len) — attention distribution
        """
        # decoder_hidden: (batch, hidden) -> (batch, 1, hidden) for broadcasting
        dec_expanded = decoder_hidden.unsqueeze(1)
        
        # Score: how relevant is each encoder state to the current decoder state?
        scores = self.v(torch.tanh(
            self.W1(dec_expanded) + self.W2(encoder_outputs)
        )).squeeze(-1)  # (batch, src_len)
        
        # Attention weights (softmax over source positions)
        weights = F.softmax(scores, dim=-1)  # (batch, src_len)
        
        # Context: weighted combination of encoder states
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        
        return context, weights

# Demo: Attention at one decoder step
attn = BahdanauAttention(hidden_dim=128)
dec_h = torch.randn(2, 128)      # Current decoder state
enc_outs = torch.randn(2, 8, 128)  # 8 encoder states

context, weights = attn(dec_h, enc_outs)

print('Bahdanau Attention Demo:')
print(f'  Decoder state:     {dec_h.shape}')
print(f'  Encoder states:    {enc_outs.shape}')
print(f'  Context vector:    {context.shape}  (weighted mix of encoder states)')
print(f'  Attention weights: {weights.shape}')
print(f'\n  Weights for batch 0: {weights[0].detach().numpy().round(3)}')
print(f'  Sum = {weights[0].sum().item():.1f} (probability distribution)')
print(f'\n  Highest attention on position {weights[0].argmax().item()}')
print(f'  → Decoder is "looking at" encoder step {weights[0].argmax().item()}')

---
## 3 · Full Seq2Seq with Attention

In [ ]:
class AttentionDecoder(nn.Module):
    """Decoder that uses attention to access all encoder states."""
    
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = BahdanauAttention(hidden_dim)
        self.lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.output_proj = nn.Linear(hidden_dim, vocab_size)
    
    def forward_step(self, token, hidden, cell, encoder_outputs):
        """One step of decoding."""
        embedded = self.embedding(token)  # (batch, embed_dim)
        
        # Attention: which encoder states matter now?
        context, attn_weights = self.attention(hidden.squeeze(0), encoder_outputs)
        
        # Concatenate context + embedded input
        lstm_input = torch.cat([embedded, context], dim=1).unsqueeze(1)
        
        # LSTM step
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        
        # Predict next token
        prediction = self.output_proj(output.squeeze(1))
        
        return prediction, hidden, cell, attn_weights

# Full Seq2Seq with Attention
encoder = Seq2SeqEncoder(5000, 64, 128)
decoder = AttentionDecoder(5000, 64, 128)

# Encode source
src = torch.randint(0, 5000, (2, 8))
enc_outputs, enc_hidden, enc_cell = encoder(src)

# Decode step by step
current_token = torch.zeros(2, dtype=torch.long)  # <SOS> token
hidden, cell = enc_hidden, enc_cell

print('Attention during translation (batch 0):')
src_words = ['The', 'cat', 'sat', 'on', 'the', 'mat', '.', '<pad>']

for step in range(4):
    pred, hidden, cell, attn_w = decoder.forward_step(
        current_token, hidden, cell, enc_outputs
    )
    current_token = pred.argmax(dim=1)
    weights = attn_w[0].detach().numpy()
    top_idx = weights.argmax()
    print(f'  Step {step}: attending to "{src_words[top_idx]}" (weight={weights[top_idx]:.3f})')

---
## 4 · From RNN-Attention to Self-Attention

### The Final Insight (Vaswani et al., 2017: "Attention Is All You Need")

| RNN + Attention (Bahdanau) | Transformer (Vaswani) |
|---------------------------|----------------------|
| Encoder uses RNN (sequential) | Encoder uses Self-Attention (parallel) |
| Attention is decoder → encoder only | Self-attention within encoder AND decoder |
| Q = decoder state, K,V = encoder states | Q, K, V all from same sequence |
| O(n) sequential steps | O(1) parallel + O(n²) attention |

### 🎓 DEEP QUESTION ANSWERED

**Q:** *What proof did the Transformer authors have that attention alone was sufficient?*

**A:** Empirical evidence showed that in RNN+Attention models, the RNN's contribution was minimal — the attention mechanism was doing most of the heavy lifting. The theory: if attention can learn arbitrary weighted combinations of input representations, and we add positional encoding to preserve order, then the RNN's recurrence is redundant. The proof was the remarkable results on WMT translation benchmarks where the recurrence-free Transformer outperformed the best RNN models while training 10x faster.

**→ For the full Transformer implementation and Self-Attention deep-dive, see DL 07.**

---

## ✅ Knowledge Check

### Q1: Bottleneck 
In vanilla Seq2Seq (no attention), what happens to translation quality as the source sentence gets longer?

<details><summary>👉 Answer</summary>

Quality degrades dramatically. The fixed-size context vector cannot compress more than ~20-30 words of information effectively. Empirically, BLEU scores drop sharply for sentences longer than 20 words. This is the exact problem that attention solves.
</details>

### Q2: Attention as search
How is Bahdanau attention similar to what happens in RAG retrieval (NB28)?

<details><summary>👉 Answer</summary>

Both are query-based retrieval: In attention, the decoder state is the query and encoder states are the documents. In RAG, the user question embedding is the query and document chunk embeddings are the Keys. Both compute similarity scores, apply softmax, and return a weighted combination. RAG is literally attention applied to a document database.
</details>

### Q3: Why not just use the last encoder state?

<details><summary>👉 Answer</summary>

The last encoder state biases toward recent input tokens (recency bias). Information about the first words has been "overwritten" by later processing steps (vanishing gradient across sequence). Attention gives equal access to ALL positions, eliminating this bias.
</details>

---

## 🔨 Practical Practice

### Exercise 1: Attention Visualization
Modify the demo to accumulate attention weights across all decoder steps. Plot a heatmap with source tokens on the x-axis and target steps on the y-axis.

### Exercise 2: Luong Attention
Implement Luong (dot-product) attention: $\text{score} = h_t^{dec} \cdot h_j^{enc}$. Compare parameter counts with Bahdanau.

### Exercise 3: Teacher Forcing
Implement teacher forcing in the decoder: during training, feed the ground-truth previous token instead of the model's prediction. Why does this speed up training but can hurt inference?

---

**Next →** NLP 05: Pre-training Objectives & BERT